# Confidence-weighted Land Surfaces

*Resources:* 

[Master Plan API Documentation](https://github.com/pscedu/cpra.mp.data/tree/main)

### MP29 ESLR scenarios & confidence weighting factors

<style>
figure {
  border: 1px #cccccc solid;
  padding: 4px;
  margin: auto;
}

figcaption {
  background-color: black;
  color: white;
  font-style: italic;
  padding: 2px;
  text-align: center;
}
</style>

From these five ensemble statistics, a confidence-weighting factor, `CWF`, can be developed (see methodology in section 1, to the left). The figures below demonstrate what the `CWF` values would be at 2035 under the SSP2-RCP4.5 climate pathway.

Figure 1, below, demonstrates the CWF methodology on an *assumed normal* distribution using the CMIP6-reported ESLR values in the Gulf from 2020 through 2035, under SSP2-RCP4.5
<figure> <img src="../images/confidence_weights/ESLR_CWF_2020-2035_normal.png" width ="800"> <figcaption>Figure 1. Hypothetical normal distribution of ESLR from 2020-2035 for the Gulf, under SSP2-RCP4.5.</figcaption></figure>

Given the skew evident in the published ensemble percentiles (which amplifies in later decades), we know that an assumption of normality is not well suited for these ESLR percentile ensembles. However, in order to apply this methodology, we assume that any skewness and non-normality of the full ensemble was properly controlled for (by NASA) when determining the ESLR ensemble percentile values at each published timestep. With this assumption, the CWF methodology can be derived from a probability density function of *any* shape and does not require an assumption of normality is strictly a function of the area under the probability density function curve.

To that end, while the hypothetical/ideal normal curve shown shown in Figure 1 is helpful in conceptualizing this approach; Figure 2, below, attempts to show how the CWF methodology is applied when the ensemble is skewed and/or non-normal.

<figure> <img src="../images/confidence_weights/ESLR_CWF_2020-2035_skewed.png" width ="800"> <figcaption>Figure 2. Skewed distribution of ESLR frrm 2020-2035 for the Gulf, under SSP2-RCP4.5.</figcaption></figure>

<!--BEGIN SECTION scenarioweights-->
| MP29 Scenario ID | CMIP6 Scenario | ESLR Percentile |  MP29 Confidence Weighting Factor |
| --- | --- | --- | --- |
| S36 | SSP2-RCP4.5 | 5th percentile ESLR | 0.085 |
| S37 | SSP2-RCP4.5 | 17th percentile ESLR | 0.25 |
| S34 | SSP2-RCP4.5 | 50th percentile ESLR | 0.33 |
| S38 | SSP2-RCP4.5 | 83rd percentile ESLR | 0.25 |
| S35 | SSP2-RCP4.5 | 95th percentile ESLR | 0.085 |
<!--END SECTION scenarioweights-->
*References:*
[Confidence-weight land methodology Miro board](https://miro.com/app/board/uXjVGoH_2cw=/?share_link_id=435753759569)


## Load and Prepare Data
Imports, parameters, database connection, and project/ecoregion scope setup.

In [1]:
import polars as pl
from polars import col
import geopandas as gpd
import rasterio as rio
import numpy as np
from cpra.mp.data import read_data
from rasterio.features import shapes
import altair as alt
from sqlalchemy import create_engine
import getpass
from IPython.display import display
from contextlib import ExitStack
import os


In [ ]:
# MP29 ESLR Scenario Weighting Factors to be used for confidence weighting

scenario_weights = {36:0.085,37:0.25,34:0.33,38:0.25,35:0.085}


model_group_id = 500
SALINITY_THRESHOLD = 5.5
FRESHWATER_MARSH_THRESHOLD = 0.5

# Standard MPD Parameters
# More Information on Dimensions: https://github.com/pscedu/cpra.mp.data/blob/main/docs/index.md
hydrocompartment_id_column = "hydrocompartment_id"
freshwater_marsh_value_column = "freshwater_marsh_value"
calendar_year_column = 'calendar_year'
salinity_variable = "sal"
freshwater_marsh_variable = "pl_fm" # percent land freshwater marsh 

# Master Plan Attribute Database (MAD) Table Names
gsd_a_hydrocompartent = "gsd.a_hydro_compartment"

# Link to Veg Hydrocompartment ID Crosswalk
crosswalk_veg_cell_hydro_compartment = "/ocean/projects/bcs200002p/shared/grids/crosswalks/veg_grid_cell_v001__hydro_compartment_v001.tif"

# Establish output filename 
username = getpass.getuser()
base_folder = f"/ocean/projects/bcs200002p/{username}"
OUTPUT_TIFF = os.path.join(base_folder, f"analysis_project_benefits_{scenario_id}_{model_group_id}.tif")

In [3]:
# MAD Connection Properties
password = getpass.getpass("Enter your password: ")
port = "5432"
host = "vm002.bridges2.psc.edu"
db_name = "mpd_dev"
uri = f"postgresql://{username}:{password}@{host}:{port}/{db_name}"
engine = create_engine(uri)

In [6]:
# Load annual freshwater marsh rasters and store tif paths for each year for efficient windowed reads in the analysis loop.
freshwater_marsh_tif_df = (
    read_data(
    variable="ffibs_coverage_ratio",
    grid="veg_grid_cell_v001",
    time_unit="annual",
    model_group_id=model_group_id,
    scenario_id=scenario_id,
    ffibs_type=freshwater_marsh_variable,
    ))

freshwater_marsh_tif_dict = dict(freshwater_marsh_tif_df.collect().select("calendar_year", "path").iter_rows())
